In [4]:
import sqlite3
import json

# ==========================================
# THAY ĐỔI PATH NÀY THÀNH PATH DB CỦA BẠN
DB_PATH = "cookpad_clean.db"
OUTPUT_PATH = "ingredients_export.json"
# ==========================================

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("SELECT id, name FROM ingredients ORDER BY id ASC")
rows = cursor.fetchall()

conn.close()

data = [{"id": row[0], "name": row[1]} for row in rows]

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"✅ Exported {len(data)} ingredients → {OUTPUT_PATH}")
print(json.dumps(data[:5], ensure_ascii=False, indent=2))  # preview 5 dòng đầu

✅ Exported 1353 ingredients → ingredients_export.json
[
  {
    "id": 7,
    "name": "măng"
  },
  {
    "id": 9,
    "name": "cà chua"
  },
  {
    "id": 10,
    "name": "xương gà"
  },
  {
    "id": 16,
    "name": "hành lá"
  },
  {
    "id": 21,
    "name": "rau răm"
  }
]


In [7]:
import sqlite3
import json

# ==========================================
# THAY ĐỔI PATH NÀY
DB_PATH = "cookpad_clean.db"
HASHMAP_PATH = "./json_compare/result.json"
# ==========================================

with open(HASHMAP_PATH, 'r', encoding='utf-8') as f:
    hashmap = json.load(f)

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = OFF")  # tắt tạm để update thoải mái
cursor = conn.cursor()

total_deprecated = 0
total_dish_updated = 0

try:
    conn.execute("BEGIN TRANSACTION")

    for canonical_name, ids in hashmap.items():
        canonical_id = ids[0]
        duplicates = ids[1:]

        print(f"\n[{canonical_name}] canonical_id={canonical_id}({canonical_name})")

        for dup_id in duplicates:
            # Lấy tên duplicate để log
            cursor.execute("SELECT name FROM ingredients WHERE id = ?", (dup_id,))
            row = cursor.fetchone()
            dup_name = row[0] if row else "NOT FOUND"

            # Update dish_ingredient: trỏ về canonical
            cursor.execute(
                "UPDATE dish_ingredient SET ingredient_id = ? WHERE ingredient_id = ?",
                (canonical_id, dup_id)
            )
            dishes_affected = cursor.rowcount

            # Deprecate ingredient
            cursor.execute(
                "UPDATE ingredients SET is_deprecated = 1 WHERE id = ?",
                (dup_id,)
            )

            total_dish_updated += dishes_affected
            total_deprecated += 1
            print(f"  ↳ {dup_id}({dup_name}) → {canonical_id} | dish_ingredient rows updated: {dishes_affected}")

    conn.commit()
    print(f"\n✅ DONE!")
    print(f"   Groups processed : {len(hashmap)}")
    print(f"   Records deprecated: {total_deprecated}")
    print(f"   dish_ingredient rows updated: {total_dish_updated}")

except Exception as e:
    conn.rollback()
    print(f"\n❌ ERROR: {e}")
    print("   Rolled back — no changes made.")
    raise

finally:
    conn.execute("PRAGMA foreign_keys = ON")
    conn.close()


[ngò rí] canonical_id=102(ngò rí)
  ↳ 64(ngò) → 102 | dish_ingredient rows updated: 282
  ↳ 2592(ngò rí) → 102 | dish_ingredient rows updated: 7
  ↳ 4617(rau ngò) → 102 | dish_ingredient rows updated: 3
  ↳ 1565(rễ mùi) → 102 | dish_ingredient rows updated: 5
  ↳ 5279(rễ ngò) → 102 | dish_ingredient rows updated: 8
  ↳ 5423(lá ngò rí) → 102 | dish_ingredient rows updated: 1
  ↳ 9224(ngò mùi) → 102 | dish_ingredient rows updated: 2

[mùi tàu] canonical_id=879(mùi tàu)
  ↳ 919(rau mùi tàu) → 879 | dish_ingredient rows updated: 10
  ↳ 4689(ngò tàu) → 879 | dish_ingredient rows updated: 1
  ↳ 3575(rau ngò gai) → 879 | dish_ingredient rows updated: 29
  ↳ 9759(mùi răng cưa) → 879 | dish_ingredient rows updated: 1

[bột ngò] canonical_id=3024(bột ngò)

[ngò tây] canonical_id=10102(ngò tây)

[húng] canonical_id=691(húng)
  ↳ 1194(húng) → 691 | dish_ingredient rows updated: 3
  ↳ 1908(lá húng) → 691 | dish_ingredient rows updated: 1
  ↳ 6679(rau húng) → 691 | dish_ingredient rows updated: 10


In [9]:
import sqlite3

# ==========================================
DB_PATH = "cookpad_clean.db"
# ==========================================

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = OFF")

try:
    conn.execute("BEGIN TRANSACTION")

    # Xóa các record deprecated
    cursor = conn.execute("DELETE FROM ingredients WHERE is_deprecated = 1")
    deleted = cursor.rowcount
    print(f"✅ Deleted {deleted} deprecated ingredients")

    # SQLite không support DROP COLUMN trực tiếp trước version 3.35
    # Cách chuẩn: tạo bảng mới không có cột is_deprecated, copy data, đổi tên
    conn.execute("""
        CREATE TABLE ingredients_new (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT UNIQUE NOT NULL,
            name_en TEXT, ingredient_type TEXT, allergen_tags TEXT,
            category TEXT, is_animal_based INTEGER, seasonal_availability TEXT,
            regional_availability TEXT, energy_density REAL, sodium_density REAL,
            hydration_score REAL, electrolyte_density REAL, satiety_score REAL,
            health_benefit_score REAL, glycemic_index REAL, thermogenic_score REAL,
            warming_score REAL, cooling_score REAL, flavor_profile TEXT
        )
    """)

    conn.execute("""
        INSERT INTO ingredients_new
        SELECT id, name, name_en, ingredient_type, allergen_tags, category,
               is_animal_based, seasonal_availability, regional_availability,
               energy_density, sodium_density, hydration_score, electrolyte_density,
               satiety_score, health_benefit_score, glycemic_index, thermogenic_score,
               warming_score, cooling_score, flavor_profile
        FROM ingredients
    """)

    conn.execute("DROP TABLE ingredients")
    conn.execute("ALTER TABLE ingredients_new RENAME TO ingredients")

    conn.commit()
    print(f"✅ Dropped column is_deprecated")
    print(f"\n✅ DONE!")

except Exception as e:
    conn.rollback()
    print(f"\n❌ ERROR: {e} — Rolled back.")
    raise

finally:
    conn.execute("PRAGMA foreign_keys = ON")
    conn.close()

✅ Deleted 233 deprecated ingredients
✅ Dropped column is_deprecated

✅ DONE!
